# HW1 Question 3: QFT on the QuEST CUDA Backend

Benchmark: `qft_bench_naive.c`. Platform: Google Colab, NVIDIA T4.

## Environment

In [ ]:
%%bash
nvidia-smi --query-gpu=name,compute_cap,memory.total --format=csv
nvcc --version | tail -2
cmake --version | head -1
which nsys || ls /usr/local/cuda*/bin/nsys /opt/nvidia/nsight-systems/*/bin/nsys 2>/dev/null || echo "nsys not found"
free -g | head -2

## Nsight Systems installation

In [ ]:
%%bash
apt-get update -qq
apt-cache search nsight-systems | head
apt-get install -y -qq nsight-systems-cli 2>&1 | tail -3 || true
which nsys || ls /opt/nvidia/nsight-systems/*/bin/nsys

## Benchmark source

In [ ]:
from google.colab import files
up = files.upload()
import os, shutil
for name in up:
    shutil.move(name, "/content/qft_bench_naive.c")
os.makedirs("/content/results", exist_ok=True)
print(open("/content/qft_bench_naive.c").read()[:300])

## Build (QuEST, CUDA backend, `-lineinfo`)

In [ ]:
%%bash
set -e
cd /content
[ -d QuEST ] || git clone --depth 1 https://github.com/QuEST-Kit/QuEST.git
cd QuEST
cp /content/qft_bench_naive.c .
CC=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d '.')
echo "compute cap -> $CC"
cmake -B build -DQUEST_ENABLE_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=$CC \
      -DCMAKE_CUDA_FLAGS="-lineinfo" -DUSER_SOURCE_NAMES=qft_bench_naive.c \
      -DUSER_OUTPUT_EXE_NAME=qft_naive
cmake --build build -j4 2>&1 | tail -5

## Q1: 24-qubit run

In [ ]:
%%bash
cd /content/QuEST
./build/qft_naive 24 5 | tee /content/results/run_24q.txt

## Dependency: GNU time

In [ ]:
!apt-get install -y -qq time

## Q2: Memory usage, 28 qubits

GPU memory: `nvidia-smi` polling (whole-GPU and per-process).
Host memory: peak RSS from `/usr/bin/time -v`, `VmRSS`/`VmHWM` from `/proc/PID/status`, and `free -m`.

In [ ]:
%%bash
R=/content/results; cd /content/QuEST
echo "== baseline =="; nvidia-smi --query-gpu=memory.used,memory.total --format=csv,noheader; free -m | head -2
/usr/bin/time -v ./build/qft_naive 28 5 > $R/run_28q.txt 2> $R/time_28q.txt &
TP=$!; sleep 0.3
PID=$(pgrep -n -f "build/qft_naive 28")
echo "t,gpu_used_MiB,proc_gpu_MiB,vmrss_MiB,vmhwm_MiB,sys_used_MiB" > $R/mem_samples_28q.csv
T0=$(date +%s.%N)
while kill -0 $TP 2>/dev/null; do
  T=$(echo "$(date +%s.%N) - $T0" | bc)
  G=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | tr -d ' ')
  PG=$(nvidia-smi --query-compute-apps=used_memory --format=csv,noheader,nounits 2>/dev/null | head -1 | tr -d ' ')
  RSS=$(awk '/VmRSS/{printf "%.0f",$2/1024}' /proc/$PID/status 2>/dev/null)
  HWM=$(awk '/VmHWM/{printf "%.0f",$2/1024}' /proc/$PID/status 2>/dev/null)
  SU=$(free -m | awk '/Mem:/{print $3}')
  echo "$T,$G,$PG,$RSS,$HWM,$SU" >> $R/mem_samples_28q.csv
  sleep 0.5
done
wait $TP
cat $R/run_28q.txt
grep -E "Maximum resident|Elapsed" $R/time_28q.txt
awk -F, 'NR>1{ if($2>g)g=$2; if($3>p)p=$3; if($4>r)r=$4; if($5>h)h=$5; if($6>s)s=$6 } END{print "peak GPU used (MiB):",g,"| per-process GPU (MiB):",p,"| peak VmRSS (MiB):",r,"| VmHWM (MiB):",h,"| peak sys used (MiB):",s}' $R/mem_samples_28q.csv

## Nsight Systems profiles (24, 16 and 28 qubits)

In [ ]:
%%bash
NSYS=$(which nsys || ls /usr/local/cuda*/bin/nsys /opt/nvidia/nsight-systems/*/bin/nsys 2>/dev/null | head -1)
echo "using $NSYS"; $NSYS --version
cd /content/QuEST
for N in 24 16 28; do
  $NSYS profile --trace=cuda,osrt --stats=true --force-overwrite=true \
        -o /content/results/qft_${N}q ./build/qft_naive $N 5 > /content/results/nsys_${N}q_stats.txt 2>&1
  echo "profiled $N qubits (exit $?)"
done

## Q3: Kernel summary (`cuda_gpu_kern_sum`), 24 qubits

In [ ]:
%%bash
cd /content/results
grep -E "SKIPPED" nsys_24q_stats.txt || echo "(kernel data captured OK)"
sed -n '/Executing .cuda_gpu_kern_sum/,/Executing .cuda_gpu_mem_time_sum/p' nsys_24q_stats.txt | cut -c1-230 | head -14

## Q6: Benchmark timing and kernel summaries, 16 / 24 / 28 qubits

In [ ]:
%%bash
cd /content/results
for N in 16 24 28; do
  echo "================ $N qubits ================"
  grep -E "rep |best / mean|time per gate|gates per QFT" nsys_${N}q_stats.txt
  sed -n '/Executing .cuda_gpu_kern_sum/,/Executing .cuda_gpu_mem_time_sum/p' nsys_${N}q_stats.txt | cut -c1-230 | head -9
done

## Results archive

In [ ]:
%%bash
cd /content && rm -f hw1_q3_results.zip && zip -qr hw1_q3_results.zip results && ls -la hw1_q3_results.zip

In [ ]:
from google.colab import files
files.download("/content/hw1_q3_results.zip")